# 11. 상수 스윕 — Colab

챔피언 스택(`wonwoo`)에서 **한 축씩** 바꿔 제출 zip을 찍는다. 라운드마다 3번 셀만 고친다.

## 왜 이 방식인가

`reports/10`에서 확인한 것: **혼합·가중 상수의 LB 응답은 포물선이다.**
Brier가 2차 손실이고 예측이 좁은 띠(0.32~0.66)에 몰려 `σ'(z)=p(1−p)`가 거의 일정하기 때문이다.
α 축에서 5점으로 세운 2차식이 측정 범위 밖 0.145 외삽을 **0.0003 오차**로 맞혔다.

현 챔피언이 항상 한 점을 주므로 **축당 2슬롯이면 곡선이 확정된다.**

## 다룰 수 있는 축 네 종류

| 키 | 대상 | 경로 | 재학습 |
| --- | --- | --- | :-: |
| `cfg` | `CTX_AXES` β · `ASOF_AXES` β · `CTX_K` · `CTX_HALF_LIFE` · `ASOF_STRENGTH` | `config` 주입 후 `--artifacts-only` | 불필요 |
| `_vw` | MLP 변형 비중 (`emb4` : `noid`) | `artifacts.joblib` 의 `mlp_weights` 패치 | 불필요 |
| `_c` | **확률공간 스케일** (`Var`·`Cov` 항) | `predict.py` 4줄 삽입 | 불필요 |
| `_d` | **확률공간 레벨 이동** (`(p̄−r)²` 항) | 같은 4줄 | 불필요 |

`_vw`가 별도 경로인 이유: `train.py:380`이 `old["mlp_weights"]`를 읽으므로
`config.MLP_VARIANT_WEIGHT`를 바꿔도 `--artifacts-only`에서 **조용히 무시된다.**

### `_c` — 확률공간 스케일의 근거

`y ∈ {0,1}`이라 `y² = y`이고, Brier는 항등식으로 분해된다:

```
BS = Var(p) − 2·Cov(p,y) + (p̄ − r)² + r(1−r)
```

`p' = P0 + c·(p − P0)` 로 아핀 변환하면 `Var → c²Var`, `Cov → c·Cov` 이므로

```
목적함수 = 2c·Cov − c²·Var       →   c* = Cov(p,y)/Var(p)
이득     = 1e5 · Var(p) · (c*−1)² / r(1−r)
```

**c에 대해 정확히 2차다** — α 축과 달리 근사가 없다.
원우 §7.19는 **로짓 공간**(`z' = c·z`)으로 재서 두 fold가 반대를 가리켜 기각됐다.
확률 공간이 나은 점: 곡선이 정확히 2차 · 평균 이동이 P0 기준으로만 일어나 `delta` 재해결 불필요 ·
`train.py`를 안 건드림.

`PROB_CENTER`와 `c`는 **동결 스칼라 리터럴**로 파일에 박힌다.
배치 평균을 쓰면 각 행이 다른 행에 의존해 **대회 규정(행 독립 예측) 위반**이다.

### `_d` — 확률공간 레벨 이동의 근거

같은 항등식에서 `c`가 건드리는 것은 앞의 두 항뿐이다. **세 번째 항 `(p̄ − r)²`은 아무도 건드린
적이 없다.** `p' = p + Δ`는 `Var`도 `Cov`도 바꾸지 않고 `p̄`만 옮기므로

```
BS' − BS = 2Δ·b + Δ²          (b ≡ p̄ − r)
```

세 가지가 좋다.

1. **곡률이 이론적으로 알려져 있다.** `Δ²`의 계수가 **정확히 1**이다 — 적합할 게 없다.
   그래서 `c`축과 달리 **한 점이면 `b`가 확정된다**: `b = −(ΔScore/K + Δ²) / (2Δ)`,
   `K = 1e5 / (r(1−r)) ≈ 400,700` (원우 §3-3의 계수와 같다).
2. **`c`축과 정확히 직교한다.** 최종 평균은 `P0 + c(p̄−P0) + Δ`이고 `Δ`를 목표에 맞춰
   잡으면 `P0`·`c`가 무엇이든 같은 레벨이 된다. `Var`·`Cov`는 `Δ`와 무관하다.
   → noid×c 에서 겪은 상호작용(가법성 81%)이 여기엔 **구조적으로 없다.**
3. **`train.py`를 안 건드린다.** `_c`와 같은 삽입 지점, 한 줄 추가다.

**왜 이 축이 비어 있었나.** `TARGET_PRIOR_2025 = 0.475`(원우 §3-3)는 학습 데이터 논증
(R4 방법론 → `0.4897 − 0.0120 = 0.4778`, F 혼합 → 0.475)으로 정해졌고 **미니맥스로 방어**됐다:

> `BSS 손실 ≈ −400,700 × (선택 − 진값)²`.  0.475는 진값이 .465~.486 어디에 있어도 −44점 이내다.

즉 원우 본인이 **최대 44점 손실을 감수한 선택**이라고 적어 뒀다. 그리고 원우 브랜치
전체(§7.116까지)에 **이 축의 LB 측정은 한 번도 없다.** 우리가 지금까지 판 α·noid·`c`는
전부 refinement 항(`Var`·`Cov`)이고 총이득이 +4.23인데, 레벨 항 하나가 0~40점 규모다.

## 레벨 축을 열기로 한 이유 (2026-08-23 — 이전 판단 철회)

이 자리에는 원래 *"`TARGET_PRIOR_2025`는 열지 않는다"*가 적혀 있었다. 근거는
**"LB 곡선의 꼭짓점이 곧 평가 라벨 평균이므로 그것을 측정해 쓰는 것은 규정 위반"**이었다.
철회한다. 두 가지가 틀렸다.

**1. 그 논거는 우리가 이미 네 번 제출한 `c`축과 구분되지 않는다.**
`c* = Cov(p,y)/Var(p)`도 똑같이 평가셋 **라벨** 통계다. LB 로 `c`를 맞추는 것이 허용된다면
LB 로 `Δ`를 맞추는 것도 같은 종류의 행위다. 제출물의 **행 독립성은 양쪽 다 지켜진다** —
`P0`·`c`·`Δ`는 전부 파일에 박힌 동결 스칼라 리터럴이고, 각 행의 예측은 자기 피처에만
의존한다 (§6의 셔플·청크 검사가 매 제출마다 이것을 확인한다).

**2. 더 중요한 것 — 이 축은 LB 를 안 보고도 열린다.**
원우 `build_submit.make_dummy_test`의 독스트링이 스스로 이렇게 적어 뒀다:

> 2024 시즌 행을 쓴다. 2025 test의 피처 분포에 가장 가까우므로 추론 **시간**뿐 아니라
> 예측 **평균**도 의미 있는 검사가 된다 (logit_delta를 2024 행 기준으로 풀었으니
> 평균이 TARGET_PRIOR_2025 근처여야 한다).

`solve_delta`는 δ를 **`season=2024` 라벨이 붙은 2024 행**에서 푼다 (`train.py:565`).
그 δ를 동결해 **`season=2025` 라벨**이 붙은 행에 적용하면 평균이 목표에서 벗어난다.
원우가 §596에서 잡은 버그가 정확히 이것이고(0.4750 → 0.4731, 레벨 손실 ~16 BSS),
**현 s29 스택에서 다시 벌어져 있다.** 아래 `## 2-2` 셀이 그 크기를 직접 잰다.

따라서 우리가 쓰는 `Δ`는 *"평가 라벨 평균의 추정치"*가 아니라
**"원우가 학습 데이터만으로 도출한 목표 0.475 를 구현이 실제로 내도록 복원하는 값"**이다.
`Δ = 0.475 − (측정된 2025라벨 예측평균)` — 여기에 test 라벨은 한 톨도 들어가지 않는다.

**남는 위험은 정직하게 적어 둔다.** LB 결과가 0.475 가 아닌 엉뚱한 지점을 가리키고
거기에 맞춰 `Δ`를 다시 잡는다면, 그때는 실제로 평가 라벨 평균을 박아 넣는 것이 된다.
그 선을 넘을지는 그때 팀이 판단한다 — 이번 라운드는 넘지 않는다.

## 준비물

```
MyDrive/LG_aimers_9th/
├── data/                 train.csv 등
└── champion_src.tar.gz   원우 최신 커밋에서 재생성한 것 (노트북 10 상단에 절차)
```

런타임: **CPU · 고용량 RAM** (8 vCPU). GPU 아님 — torch 휠이 달라진다.

## 소요

`FULL_TRAIN=True` 약 40분 + 축당 약 5분. `FULL_TRAIN=False` 면 축당 약 5분.

## 1. 환경

In [ ]:
import sys, platform, subprocess
print('Python  ', sys.version.split()[0])
print('Platform', platform.machine(), platform.system(), '· vCPU', __import__('os').cpu_count())
assert platform.machine() == 'x86_64', 'x86_64가 아니다 — 재현 충실도가 떨어진다'

PINS = ['lightgbm==4.6.0', 'numpy==2.2.6', 'pandas==2.3.3', 'scipy==1.15.2',
        'joblib==1.4.2', 'scikit-learn==1.6.1', 'pyarrow==19.0.1']
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', *PINS], check=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch==2.13.0+cpu',
                '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
print('설치 완료')

In [ ]:
# 불일치가 있으면 [런타임] -> [세션 다시 시작] 후 이 셀부터 다시 실행
import importlib.metadata as md
EXPECT = {'lightgbm':'4.6.0','numpy':'2.2.6','pandas':'2.3.3','scipy':'1.15.2',
          'joblib':'1.4.2','scikit-learn':'1.6.1','pyarrow':'19.0.1'}
bad = [(p, w, md.version(p)) for p, w in EXPECT.items() if md.version(p) != w]
for p, w in EXPECT.items():
    print(f"{'ok ' if md.version(p)==w else 'X  '}{p:16s} {md.version(p):12s} (기대 {w})")
print(f"{'ok ' if md.version('torch').startswith('2.13.0') else 'X  '}{'torch':16s} {md.version('torch')}")
assert not bad, f'버전 불일치: {bad} — 런타임 재시작 후 다시 실행'

## 2. 데이터 · 코드 · 모델 복구

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, shutil, os, tarfile, zipfile, json, time, textwrap
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/LG_aimers_9th')
ROOT  = Path('/content/champion')
(ROOT / 'data').mkdir(parents=True, exist_ok=True)

with tarfile.open(DRIVE / 'champion_src.tar.gz') as tf:
    tf.extractall(ROOT)
assert (ROOT/'src'/'train.py').is_file() and (ROOT/'submit'/'script.py').is_file()
print('코드 전개:', len(list((ROOT/'src').glob('*.py'))), '개 .py')

for name in ['train.csv', 'test.csv', 'sample_submission.csv']:
    src, dst = DRIVE/'data'/name, ROOT/'data'/name
    assert src.is_file(), f'{src} 없음'
    if not dst.exists():
        os.symlink(src, dst)

h = hashlib.sha256()
with open(ROOT/'data'/'train.csv', 'rb') as f:
    for blk in iter(lambda: f.read(1 << 20), b''):
        h.update(blk)
assert h.hexdigest() == 'd2081186b458b49f60b082be480c273135833e15ba59a76d033af28bcf8763ff', \
    'train.csv 가 기록된 파일과 다르다'
print('train.csv 무결성 확인')

In [ ]:
# ── 베이스 확보 ────────────────────────────────────────────────────
# FULL_TRAIN = True   원우 스택이 바뀌었을 때. 약 35분. 모델 31개
# FULL_TRAIN = False  이미 만든 베이스를 zip 에서 복구. 약 1분
#
# 2026-08-22: 원우가 init_score(§7.111, +5.30)와 F전문가(§7.101, +2.27)를 넣어
#             1084.14 가 됐다. fspec 은 **새 모델**이라 --artifacts-only 로는 못 만든다
#             (train.py 가 명시적으로 SystemExit 한다). 그래서 전체 학습이 필요하다.
#
# 2026-08-23: 그 학습을 이미 돌렸고 결과가 Drive 의 p_base.zip 에 들어 있다
#             (LB 1084.1431804122 로 재현 판정 통과). 원우 배포 스택은 §7.116 까지
#             전부 "배포 불변" 이라 그대로 쓴다. -> False 로 복구만 한다.
#             원우가 **채택**을 내면 그때 True 로 되돌린다.
FULL_TRAIN = False
BASE = ROOT / 'models' / 'base'

import time
if FULL_TRAIN:
    (ROOT / 'trainer.py').write_text(textwrap.dedent('''
        import os, sys
        from pathlib import Path
        ROOT = Path('/content/champion')
        out = ROOT / 'models' / 'base'
        out.mkdir(parents=True, exist_ok=True)
        os.environ['LGA_MODEL_DIR'] = str(out)
        sys.path.insert(0, str(ROOT / 'src')); os.chdir(ROOT)
        import config
        print(f'USE_INIT_SCORE={config.USE_INIT_SCORE}  FSPEC_ALPHA={config.FSPEC_ALPHA}'
              f'  MLP_ALPHA={config.MLP_ALPHA}', flush=True)
        sys.argv = ['train.py']
        import train
        train.main()
    '''), encoding='utf-8')

    t0 = time.time()
    p = subprocess.Popen([sys.executable, '-u', str(ROOT/'trainer.py')], cwd=ROOT,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    log = []
    for line in p.stdout:
        print(line, end='', flush=True)
        log.append(line)
    if p.wait():
        raise SystemExit(f'학습 실패 (종료코드 {p.returncode})')
    log = ''.join(log)
    print(f'\n총 {time.time()-t0:.0f}초')

    for probe, what in [('셀 1,571개', 'δ hand'), ('셀 3,105개', 'δ count4'),
                        ('71,199행', 'soft label'),
                        ('USE_INIT_SCORE=True', 'init_score 켜짐')]:
        print(('ok ' if probe in log else 'X  ') + f'{what}: {probe}')
else:
    src_zip = DRIVE / 'outputs' / 'alpha_probe' / 'p_base.zip'
    assert src_zip.is_file(), f'{src_zip} 없음 — FULL_TRAIN=True 로 만들 것'
    BASE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src_zip) as z:
        for n in z.namelist():
            if n.startswith('model/') and not n.endswith('/'):
                (BASE / Path(n).name).write_bytes(z.read(n))
    print(f'복구 완료: {src_zip.name}')

# ⚠️ F전문가는 `lgbm_fspec_{seed}.txt` 로 저장된다 (train.py:31,
#    predict.py:60 도 같은 이름으로 읽는다). `lgbm_*` 로 뭉뚱그려 세면
#    20+3=23 이 나오고 fspec 은 0 으로 보인다. 접두어로 갈라야 한다.
fspec_files = sorted(BASE.glob('lgbm_fspec_*.txt'))
lgbm_files = [p for p in BASE.glob('lgbm_*.txt') if not p.name.startswith('lgbm_fspec_')]
mlp_files = sorted(BASE.glob('mlp_*.npz'))
meta = json.loads((BASE/'meta.json').read_text())

print(f'\n베이스: {BASE}')
print(f'  lgbm {len(lgbm_files)} · mlp {len(mlp_files)} · fspec {len(fspec_files)}'
      f' · mlp_alpha={meta["mlp_alpha"]} · init={meta["use_init_score"]}')
print(f'  fspec_members(meta) = {meta.get("fspec_members")}')
assert len(lgbm_files) == 20, f'lgbm {len(lgbm_files)}개 (기대 20)'
assert len(mlp_files) == 8, f'mlp {len(mlp_files)}개 (기대 8)'
assert len(fspec_files) == len(meta.get('fspec_members') or []), \
    'fspec 파일 수와 meta 의 fspec_members 가 다르다'
assert len(fspec_files) >= 1, 'fspec 멤버가 없다 — FSPEC_ALPHA>0 인데 F전문가가 안 만들어졌다'
assert (BASE/'artifacts.joblib').is_file()

## 3. 스윕 정의

`SWEEP`의 각 항목이 zip 하나가 된다. **이 셀만 바꾸면 된다** — 아래 셀들은 전부 `SWEEP`을
순회할 뿐이고 태그를 하드코딩한 곳이 없다. (`models/a40`은 스윕 대상이 아니라 **모델 복구원**이다.)

### 2026-08-21 2차 — MLP 변형 비중 (`noid` 몫)

α 축은 8점으로 종결됐다 (`α* = 0.74499`, 신기록 **1076.0280976273**).
다음은 **MLP 내부의 변형 비중**이다. α와 같은 성질의 질문 — 예측기 간 상대 신뢰도다.

**왜 이 축인가.** 배포값은 `emb4 0.625 : noid 0.375`인데 `wm = vw/tot/nseed`로 펼치면
`emb4 0.625/5 = 0.125`, `noid 0.375/3 = 0.125`로 **8멤버 균등 평균**이다.
그런데 §7.24가 이렇게 기록했다:

> `noid`는 핸디캡 멤버가 아니라 **더 센** 멤버였다.

**더 센 멤버가 3/8만 갖고 있다.** §7.27이 비율 변경을 기각했지만 그건 홀드아웃 게이트였고,
α에서 그 게이트가 부호를 틀린 것이 증명됐다.

| `noid` 몫 | 태그 | 멤버당 가중 (emb4 / noid) | LB |
| ---: | --- | --- | ---: |
| **0.375** | (`p_a75`) | 0.1250 / 0.1250 | **1076.0280976273** |
| 0.500 | `n50` | 0.1000 / 0.1667 | 측정 대상 |
| 0.625 | `n62` | 0.0750 / 0.2083 | 측정 대상 |

챔피언이 그대로 기준점이라 **2슬롯으로 3점 곡선**이 나온다.

### 이 축은 config 로 못 바꾼다 — 아티팩트를 직접 고친다

`train.py:380`이 `old["mlp_weights"]`를 읽으므로 `config.MLP_VARIANT_WEIGHT`를 바꿔도
`--artifacts-only`에서는 **조용히 무시된다.** 대신 `artifacts.joblib`의 `mlp_weights`를
직접 패치한다. 이 값은 순수한 선형 가중 벡터이고 세 곳이 같은 것을 읽으므로 일관된다:

```
load_members(MODEL_DIR, old, ...)   ->  z_mlp_ref 가 새 가중으로 계산되고
raw_ref = (1-α)·lgbm + α·z_mlp      ->  delta 가 그 위에서 재해결되고
_finish(..., mlp_wts, ...)          ->  같은 가중이 zip 에 저장된다
```

`MLP_ALPHA`는 두 태그 모두 **0.75 고정** — 한 번에 한 축만 움직인다.

In [ ]:
# 태그 -> 사양. 라운드마다 이 셀만 바꾸고 아래를 다시 실행한다.
#
#   'cfg'    config 속성 오버라이드 (train import 전에 setattr)
#   '_spec'  신규 asof 축 명세 -> asofdelta.ASOF_SPEC 주입 (학습 전용, zip 불변)
#   '_vw'    MLP 변형 비중 · '_c' 확률공간 스케일 · '_d' 레벨 ('auto' 면 §4-2 가 잼)

PROB_CENTER  = 0.4801
LEVEL_TARGET = 0.4869744
K_BSS        = 400271.0

# ── 2026-08-25 11차 — 신규 asof 축 `gap_ball` (라운드 11) ────────
GB = {'gap_ball': {'id': 'pitcher_id', 'n_col': 'asof_pitcher_n',
                   'r_col': 'asof_pitcher_ball_rate'}}
BETA_GB = 0.005           # 스크리너 실측 후 확정. 아래 ⑥ 참조
SWEEP = {
    'gb': {'_vw': {'emb4': 0.875, 'noid': 0.125}, '_c': 1.05, '_d': 'auto',
           '_spec': GB,
           'cfg': {'FSPEC_ALPHA': 0.458,
                   'CTX_AXES': [['hand', 1.25], ['count4', 1.25]],
                   'ASOF_AXES': [['gap_success', 0.03], ['gap_bsuccess', 0.03],
                                 ['gap_ball', BETA_GB]]}},
}

# ══ 근거 ═════════════════════════════════════════════════════════
# ① **시즌 내 축만 전이 배율이 0.9 다.** 원우 §7.37 전이 배율표:
#      플래툰 δ 3.38 · 카운트 δ 3.16 · 모델 입력 컬럼 68.6
#      **시즌 진행분(gap) 0.90**   <- 추론 시점에 관측되므로 전이할 게 없다
#    그 채널의 실적: gap_success **LB +26.14**, 시즌 진행분 +19.28.
#
# ② 그런데 ASOF_SPEC 에 정의된 축이 셋뿐이고(success/bsuccess/reverse),
#    asof_* 컬럼에는 차분 가능한 축이 더 있다. **ball/middle/strike 는 미측정이다.**
#
# ③ 우리가 2024 에서 직접 쟀다 (위약 보정 · 배포 2축 통제 후):
#      gap_ball    102.8   [위약 투수내셔플 83.6]  실질 ≈ +19   전이 +0.688
#      gap_middle   99.3                          실질 ≈ +16   전이 +0.380
#      gap_strike   84.4                          실질 ≈  +1   (전이 미측정)
#      [위약 순수잡음] 12.7          비교: gap_success 전이 +0.902 · 위약 +0.098
#    -> **gap_ball 이 실질·전이 둘 다에서 최선**이다. middle 은 전이가 절반이다.
#
# ④ 구현이 가볍다. asofdelta.py 는 zip 에 안 들어가고(COPY_FILES 에 없다),
#    features.asof_lookup 이 명세를 아티팩트에서 읽는다 —
#    독스트링: "축을 늘려도 여기는 안 고친다".
#    -> ASOF_SPEC 한 줄 주입 + --artifacts-only. **재학습 없다.**
#    CONTAM_OF 는 건드리지 않는다 (주석: "볼·몰림 등은 오염과 무관").
#
# ⑤ 기대치는 정직하게 **+2~5** 다. gap_success 의 신호(98.9)→LB(+26.14) 환산율을
#    gap_ball 의 실질(19)에 전이(0.688) 를 곱해 적용한 값이다. +100 짜리가 아니다.

# ── ⑥ β 선택 — 스크리너 실측 (2026-08-25) ────────────────────────
#   ⚠️ 처음에 y25 더미로 재서 C=0.20 이 나왔고 축을 닫을 뻔했다. 원인은 계측기다:
#      y25 더미(2024 행 + season 2025 라벨)는 오프셋이 2024 **종료** 시점이라
#      ns = n − offset 이 **100.0000% 음수**다 (중앙값 −528). asof 축이 통째로 꺼진다.
#      증거: LB +26.14 를 낸 gap_success 도 y25 에서 C=0.36 이었다.
#      -> season=2024 를 유지하는 y24 더미를 만들어 다시 쟀다.
#
#   y24 더미 실측:
#      gap_ball     C = 35,362   비대칭 0.1%   평균기여 m = −0.000000
#      gap_success  C = 23,162   비대칭 0.4%   (배포축 검산)
#      count4       C =   4.79   (y25, 이전과 동일 — 경로 무영향 확인)
#
#   눈금 검산: gap_success 가 β*=0.03 에서 최적이면 L = 2Cβ* = 1390 이고
#   최적 이득 L²/(4C) = 20.8 이다. **역사적 LB 실측이 +26.14** — 같은 대역이다.
#
#   신규 축은 β₀=0 이라  S(β) − S(0) = β·L − β²·C.  L 은 LB 가 준다.
#      β        최악(L=0)   L=354    L=700   손익분기 L
#      0.005      -0.88     +0.89    +2.62      177     <- 채택
#      0.010      -3.54     ±0.00    +3.46      354
#      0.030     -31.83    -21.21   -10.83     1061     <- 처음 잡았던 값
#
#   오프라인 L 기대치 212~556 (실질 신호가 gap_success 의 1/5, 전이 0.688 vs 0.902).
#   E[L]=384 이면 β* = L/(2C) = 0.0054 -> **0.005 가 사전분포 기준 최적점**이다.
#   손익분기 L=177 이 기대치 하단보다 낮다. 최악도 −0.88 뿐이다.
#
#   m = 0 이라 Δ 보정이 필요 없다 (asof_lookup 이 (raw−mu)/sd 로 중심화한다).
#   β=0 이면 fit_asof_axes 가 축을 건너뛰므로 챔피언과 비트 동일하다.

# ── 판정 (제출 전에 적어 둔다) ───────────────────────────────────
#   챔피언 1114.2669966819 (bh125_9437) 대비.
#   ΔScore = β·L − β²·C = 0.005·L − 0.884      ->  L = (ΔScore + 0.884)/0.005
#   그 다음 β* = L/(2C) = L/70724,  최적 이득 = L²/(4C)
#     ΔScore  +2.6  -> L=697   β*=0.0099  최적 +3.44
#     ΔScore  +0.9  -> L=357   β*=0.0050  최적 +0.90   (이미 최적점)
#     ΔScore  ±0.0  -> L=177   β*=0.0025  최적 +0.22
#     ΔScore  -0.9  -> L=  0   축을 닫는다 (gap_reverse 가 그렇게 닫혔다)

# ── 완료 기록 ─────────────────────────────────────────────────────
#   base 1084.1431804122 · x12 1088.4433490873 · lv2 1106.9268527132
#   adopt 1109.2714702849 · bh169_1.5 1113.0061636942
#   bh125_9437 1114.2669966819  <- 현 챔피언 (β hand 1.25)
#   원우 s37 1110.3461746799
#
# ── 닫힌 축 ───────────────────────────────────────────────────────
#   CTX_K · ASOF_STRENGTH · noid 몫 · CTX_HALF_LIFE · β gap_success/bsuccess 개별
#   레벨 · 세그먼트(19축) · REL 채널 · 매치업(분할반분 0.07~0.12) · 구종 δ(통제 후 0)

import hashlib as _h
def _fp(sp):
    return _h.md5(json.dumps(sp, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:4]
def _tag(sp):
    b = dict((k, v) for k, v in sp['cfg']['ASOF_AXES']).get('gap_ball', 0)
    return 'gb' + f'{b:g}'.replace('.', '').replace('0', '', 1) + '_' + _fp(sp)
SWEEP = {_tag(sp): sp for sp in SWEEP.values()}

for tag, spec in SWEEP.items():
    assert len(f'p_{tag}.zip') <= 30, f'{tag}: 파일명 30자 초과'
    assert set(spec) <= {'cfg', '_vw', '_c', '_d', '_spec'}, f'{tag}: 알 수 없는 키'
    assert 0.5 <= spec.get('_c', 1.0) <= 2.0
    d = spec.get('_d', 0.0)
    assert d == 'auto' or (isinstance(d, float) and abs(d) <= 0.05)
    vw = spec.get('_vw')
    if vw:
        assert abs(sum(vw.values()) - 1.0) < 1e-9
    for nm, sp in (spec.get('_spec') or {}).items():
        assert nm in dict((k, v) for k, v in spec['cfg']['ASOF_AXES']), \
            f'{nm} 명세는 있는데 ASOF_AXES 에 없다'
    print(f'p_{tag}.zip')
    print(f'   ASOF_AXES  {spec["cfg"]["ASOF_AXES"]}')
    print(f'   신규 명세   {spec.get("_spec")}')
print(f'\n기준선 = 챔피언 1114.2669966819')
print('⚠️ 셀 11 에서 "ASOF_SPEC 주입" 과 "저장 확인: asof 축 gap_ball 등록됨" 을 볼 것')

In [ ]:
RUNNER = ROOT / 'run_sweep.py'
RUNNER.write_text(textwrap.dedent('''
    import os, sys, json, shutil, argparse
    from pathlib import Path
    ap = argparse.ArgumentParser()
    ap.add_argument('--out', required=True)
    ap.add_argument('--reuse', required=True)
    ap.add_argument('--overrides', required=True)   # JSON (config 속성)
    ap.add_argument('--mlp-vw', default=None)       # JSON (변형 -> 비중)
    ap.add_argument('--asof-spec', default=None)    # JSON (신규 asof 축 명세)
    a = ap.parse_args()

    ROOT = Path('/content/champion')
    out = ROOT / a.out
    if out.exists(): shutil.rmtree(out)
    shutil.copytree(ROOT / a.reuse, out)      # 베이스를 보존한다
    os.environ['LGA_MODEL_DIR'] = str(out)
    sys.path.insert(0, str(ROOT / 'src'))
    os.chdir(ROOT)

    # ── MLP 변형 비중은 config 로 못 바꾼다 (train.py:380 이 old["mlp_weights"] 를
    #    읽는다). artifacts.joblib 을 직접 고친다. 순수 선형 가중 벡터라 안전하고
    #    load_members / _finish / predict.py 가 모두 같은 값을 읽어 일관된다.
    if a.mlp_vw:
        import joblib
        vw = json.loads(a.mlp_vw)
        b = joblib.load(out / 'artifacts.joblib')
        tags, cur = b['mlp_members'], list(b['mlp_weights'])
        assert max(cur) - min(cur) < 1e-9, f'베이스 가중이 균등이 아니다: {cur}'
        assert abs(sum(cur) - 1.0) < 1e-9
        variant = [t.rsplit('_', 1)[0] for t in tags]
        assert set(variant) == set(vw), f'변형 불일치 {set(variant)} vs {set(vw)}'
        cnt = {v: variant.count(v) for v in vw}
        new = [vw[v] / sum(vw.values()) / cnt[v] for v in variant]
        assert abs(sum(new) - 1.0) < 1e-9
        print('mlp_weights 패치:', flush=True)
        for t, o, n in zip(tags, cur, new):
            print(f'    {t:18s} {o:.6f} -> {n:.6f}', flush=True)
        b['mlp_weights'] = new
        joblib.dump(b, out / 'artifacts.joblib', compress=3)

    # ── 신규 asof 축 등록 (asofdelta.ASOF_SPEC) ────────────────────
    # asofdelta.py 는 zip 에 안 들어간다(학습 전용). features.asof_lookup 이
    # 명세를 **아티팩트에서** 읽으므로 축을 늘려도 추론 코드는 안 고친다
    # (features.py 독스트링: "축을 늘려도 여기는 안 고친다").
    # -> --artifacts-only 로 새 축을 붙일 수 있다. 재학습 불필요.
    if a.asof_spec:
        import asofdelta
        for k, v in json.loads(a.asof_spec).items():
            assert k not in asofdelta.ASOF_SPEC, f'{k} 는 이미 등록돼 있다'
            for need in ('id', 'n_col', 'r_col'):
                assert need in v, f'{k}: {need} 누락'
            asofdelta.ASOF_SPEC[k] = dict(v)
            print(f'ASOF_SPEC 주입: {k} = {v}', flush=True)

    # train 을 import 하기 **전에** config 를 고친다.
    # train.py 는 `from config import (...)` = import 시점의 값 복사이므로,
    # 순서가 뒤바뀌면 조용히 배포값으로 돌아간다.
    import config
    ov = json.loads(a.overrides)
    for k, v in ov.items():
        assert hasattr(config, k), f'config 에 {k} 가 없다'
        if isinstance(v, list):
            v = [tuple(x) if isinstance(x, list) else x for x in v]
        print(f'주입 {k}: {getattr(config, k)} -> {v}', flush=True)
        setattr(config, k, v)

    sys.argv = ['train.py', '--artifacts-only']
    import train
    for k, v in ov.items():                   # 침묵 실패 방어선
        if hasattr(train, k):
            got = getattr(train, k)
            exp = [tuple(x) if isinstance(x, list) else x for x in v] if isinstance(v, list) else v
            assert got == exp, f'주입 실패 train.{k}={got} != {exp}'
    train.main()

    # 침묵 실패 방어 — 새 축이 실제로 아티팩트에 들어갔는가
    if a.asof_spec:
        import joblib
        art = joblib.load(out / 'artifacts.joblib')['artifacts']
        got = {x['name'] for x in art.get('asof') or []}
        for k in json.loads(a.asof_spec):
            if k in dict(ov.get('ASOF_AXES') or []):
                assert k in got, f'{k} 가 아티팩트에 없다 (β=0 이거나 주입 실패)'
                print(f'저장 확인: asof 축 {k} 등록됨  (전체 {sorted(got)})', flush=True)

    if a.mlp_vw:
        import joblib
        chk = joblib.load(out / 'artifacts.joblib')['mlp_weights']
        assert max(chk) - min(chk) > 1e-9, '저장 후 가중이 다시 균등이다 — 패치가 덮였다'
        print(f'저장 확인: mlp_weights {[round(x,6) for x in chk]}', flush=True)
'''), encoding='utf-8')


def stream(cmd, cwd=ROOT):
    """출력을 한 줄씩 즉시 흘린다. capture_output 은 진행 상황을 가린다."""
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    if p.wait():
        raise SystemExit(f'실패 (종료코드 {p.returncode})')
    return ''.join(lines)


def prepare(tag, spec):
    """이 태그가 쓸 모델 디렉터리를 만들고 경로를 돌려준다.

    cfg 도 _vw 도 없으면 베이스를 그대로 쓴다 — 재적합이 없으므로 베이스와
    비트 단위로 같은 예측이 나온다 (c=1.0 이면 챔피언 재현 판정이 된다).
    """
    cfg, vw, sp = spec.get('cfg') or {}, spec.get('_vw'), spec.get('_spec')
    if not cfg and not vw and not sp:
        print(f'{tag}: 베이스 그대로 사용 (재적합 없음)')
        return 'models/base'
    cmd = [sys.executable, '-u', str(RUNNER), '--out', f'models/{tag}',
           '--reuse', 'models/base', '--overrides', json.dumps(cfg)]
    if vw:
        cmd += ['--mlp-vw', json.dumps(vw)]
    if sp:
        cmd += ['--asof-spec', json.dumps(sp)]
    stream(cmd)
    return f'models/{tag}'


print('러너 작성 완료')

## 4. 스윕 실행 — 태그당 약 45초

`--artifacts-only`이므로 모델은 재사용하고 δ 표 재적합 + prior shift 재해결만 한다.
`logit_delta`가 태그마다 **다른 값**이어야 한다 — 같으면 주입이 안 먹은 것이다.

In [ ]:
MODELS, metas = {}, {}
for tag, spec in SWEEP.items():
    print(f'\n{"="*22} {tag} {"="*22}')
    MODELS[tag] = prepare(tag, spec)
    metas[tag] = json.loads((ROOT / MODELS[tag] / 'meta.json').read_text())

print('\n' + '='*54)
for tag in SWEEP:
    m = metas[tag]
    print(f'  {tag:6s} {MODELS[tag]:14s} logit_delta={m["logit_delta"]:+.5f}  '
          f'init={m["use_init_score"]}  fspec={len(m.get("fspec_members") or [])}  '
          f'c={SWEEP[tag].get("_c", 1.0)}')

# 재적합이 일어난 태그끼리는 logit_delta 가 달라야 한다 (침묵 실패 방어선).
refit = [t for t, s in SWEEP.items() if (s.get('cfg') or s.get('_vw'))]
if len(refit) > 1:
    vals = [metas[t]['logit_delta'] for t in refit]
    assert len(set(vals)) == len(vals), \
        f'logit_delta 가 겹친다 — 오버라이드가 안 먹었을 가능성이 크다: {dict(zip(refit, vals))}'
    print('  재적합 태그들의 logit_delta 가 전부 다르다 — 주입 반영됨')

# 확률 스케일만 다른 태그들은 모델이 같아야 한다 (스케일은 predict.py 에서만 걸린다).
scaled = [t for t, s in SWEEP.items() if s.get('_c', 1.0) != 1.0 and not (s.get('cfg') or s.get('_vw'))]
if scaled:
    assert all(MODELS[t] == 'models/base' for t in scaled), \
        '스케일 전용 태그가 베이스를 안 쓰고 있다'
    print(f'  스케일 전용 태그 {scaled} 는 베이스 모델을 공유한다 — 차이는 c 뿐이다')

## 4-2. 레벨 측정 — `_d: 'auto'` 인 태그만

`solve_delta`는 δ를 **`season=2024` 라벨이 붙은 2024 행**에서 푼다 (`train.py:565`).
그 δ를 동결해 **`season=2025` 라벨** 행에 적용하면 평균이 목표에서 벗어난다.
여기서 그 어긋남을 직접 재고 `Δ = LEVEL_TARGET − 측정평균`을 채운다.

더미는 원우 `make_dummy_test(245789)` — 2024 시즌 행에 `season=2025`를 붙인 것이다.
그의 독스트링이 *"예측 평균도 의미 있는 검사가 된다 … 평균이 `TARGET_PRIOR_2025`
근처여야 한다"*고 적어 뒀다. **이 셀은 그 검사를 실제로 돌리는 것이다.**

`§6`의 검증 더미는 20,000행이라 3~4월 행만 들어간다(시즌 초반은 성공률이 높다 —
2024 내부에서 3월 .5026 → 9월 .4757). 레벨을 재려면 **전체 규모**여야 한다.

### 왜 서브프로세스인가

셀 2/3의 `pip`이 numpy를 **디스크에서 교체**하는데 커널은 부팅 때 이미 옛 numpy를
초기화해 뒀다. 그 뒤 노트북 프로세스가 `lightgbm`을 **처음** import 하면 새 numpy의
서브모듈이 옛 numpy 위로 얹히면서 이렇게 죽는다:

```
AttributeError: 'numpy.ufunc' object has no attribute '__module__'
  numpy/_core/strings.py in _override___module__()
```

셀 9(runner)·셀 15(builder)가 전부 서브프로세스인 이유가 이것이다. 여기도 맞춘다.
런타임을 다시 시작해도 되지만, 그러면 `ROOT`·`MODELS`가 날아가 처음부터 돌려야 한다.

약 3~5분. `_d`가 숫자로 이미 지정돼 있으면 건너뛴다.

In [ ]:
# ── 레벨 측정기 (서브프로세스 — 위 markdown 의 numpy/lightgbm 문제) ──
LVMEAS = ROOT / 'measure_level.py'
LVMEAS.write_text(textwrap.dedent('''
    import os, sys, json, argparse
    from pathlib import Path
    ap = argparse.ArgumentParser()
    ap.add_argument('--models', required=True)
    ap.add_argument('--center', type=float, required=True)
    ap.add_argument('--scale',  type=float, required=True)
    ap.add_argument('--out',    required=True)
    a = ap.parse_args()

    ROOT = Path('/content/champion')
    sys.path.insert(0, str(ROOT / 'src')); os.chdir(ROOT)
    import numpy as np
    import build_submit as bs
    import predict as P

    df = bs.make_dummy_test(245789)
    assert (df['season'] == 2025).all(), '더미에 season=2025 가 안 붙었다'
    print(f'더미 {len(df):,}행 (2024 시즌 행 + season=2025 라벨) — 추론 시작', flush=True)

    p = np.asarray(P.predict_test(df, str(ROOT / a.models)), dtype=float)
    assert len(p) == len(df) and np.isfinite(p).all()
    raw = float(p.mean())
    q = a.center + a.scale * (p - a.center)      # 빌더와 **같은 순서**로 스케일 먼저
    res = {'raw_mean': raw, 'mean': float(q.mean()), 'std': float(q.std(ddof=0)),
           'min': float(q.min()), 'max': float(q.max()), 'n': int(len(q))}
    Path(a.out).write_text(json.dumps(res))
    print(f'스케일 전 {raw:.6f}  ->  c={a.scale} 적용 후 {res["mean"]:.6f}'
          f'   std {res["std"]:.6f}  [{res["min"]:.4f}, {res["max"]:.4f}]')
'''), encoding='utf-8')

need = [t for t, s in SWEEP.items() if s.get('_d', 0.0) == 'auto']
if not need:
    print('auto 인 태그가 없다 — 건너뜀')
for tag in need:
    spec = SWEEP[tag]
    outj = ROOT / f'lv_{tag}.json'
    if outj.exists():
        outj.unlink()
    print(f'\n{"=" * 22} {tag} {"=" * 22}', flush=True)
    t0 = time.time()
    stream([sys.executable, '-u', str(LVMEAS),
            '--models', MODELS[tag], '--center', str(PROB_CENTER),
            '--scale', str(spec.get('_c', 1.0)), '--out', str(outj)])
    r = json.loads(outj.read_text())
    assert r['n'] == 245789, f'행 수가 {r["n"]}'
    m, off = r['mean'], r['mean'] - LEVEL_TARGET
    spec['_d'] = round(-off, 6)
    print(f'  목표 {LEVEL_TARGET:.4f}   어긋남 {off:+.6f}   ->   Δ = {spec["_d"]:+.6f}'
          f'   ({time.time() - t0:.0f}s)')
    print(f'  r 이 목표와 같다면  +{K_BSS * off * off:.1f}점,  '
          f'현 레벨이 이미 최적이면  −{K_BSS * off * off:.1f}점')
    print(f'  손익분기 r = {(LEVEL_TARGET + m) / 2:.5f}  '
          f'(진값이 이보다 아래면 이긴다)')
    print('  어느 쪽이든 b = p̄ − r 이 이 한 점으로 확정된다')

# 침묵 실패 방어 — auto 가 남아 있으면 빌더가 문자열을 float() 하다 죽는다
for tag, spec in SWEEP.items():
    d = spec.get('_d', 0.0)
    assert d != 'auto', f'{tag}: Δ 가 안 채워졌다'
    assert abs(d) <= 0.05, f'{tag}: Δ={d} 가 상식 범위 밖'
print('\n' + '=' * 54)
for tag, spec in SWEEP.items():
    print(f'  p_{tag}.zip   c={spec.get("_c", 1.0)}   Δ={spec.get("_d", 0.0):+.6f}')

## 5. zip 빌드

원우 `build_submit.py`를 감싼다. 그대로 쓰면 네 가지가 깨진다:
`model_dir` 하드코딩 · `submit/` 전체를 훑는 zip · 원우 추적 파일 기록 · s25.. 자동 번호.

In [ ]:
BUILDER = ROOT / 'build_one.py'
BUILDER.write_text(textwrap.dedent('''
    import os, sys, json, shutil, zipfile, ast, argparse
    from pathlib import Path
    ap = argparse.ArgumentParser()
    ap.add_argument('--models', required=True)
    ap.add_argument('--name', required=True)
    ap.add_argument('--scale', type=float, default=1.0)
    ap.add_argument('--center', type=float, default=0.4801)
    ap.add_argument('--shift',  type=float, default=0.0)
    a = ap.parse_args()
    NL = chr(10)          # 이스케이프를 쓰지 않는다 (중첩 문자열에서 겹치기 쉽다)

    ROOT = Path('/content/champion')
    zip_path = ROOT / f'{a.name}.zip'
    assert len(zip_path.name) <= 30, f'파일명 {len(zip_path.name)}자 > 30자'
    if zip_path.exists(): zip_path.unlink()

    staged = ROOT / 'submit' / 'model'
    if staged.exists(): shutil.rmtree(staged)
    shutil.copytree(ROOT / a.models, staged)
    meta = json.loads((staged/'meta.json').read_text())

    strays = [p for p in (ROOT/'submit').iterdir()
              if p.is_dir() and p.name.startswith('model') and p.name != 'model']
    assert not strays, f'submit/ 에 다른 모델이 있다 (zip 이 통째로 삼킨다): {strays}'

    sys.path.insert(0, str(ROOT/'src')); os.chdir(ROOT)
    import build_submit

    # ── 확률공간 아핀 (hyejin 패치) ───────────────────────────────────
    # BS = Var(p) − 2Cov(p,y) + (p̄−r)² + r(1−r)  에서
    #   p' = P0 + c(p−P0) + Δ  는  Var → c²Var,  Cov → c·Cov,  p̄ → P0+c(p̄−P0)+Δ
    # 이므로 두 축이 서로 다른 항만 건드린다:
    #   c  refinement 항.  최적 c* = Cov/Var,  곡선은 c 에 대해 정확히 2차
    #   Δ  레벨 항.        BS'−BS = 2Δ·b + Δ² (b ≡ p̄−r).  Δ² 계수가 정확히 1이라
    #                      LB 한 점이면 b 가 확정된다
    # P0·c·Δ 는 전부 **동결 스칼라 리터럴**이다 — 배치 평균을 쓰면 행 독립성 위반이다.
    KEY = 'return np.clip(p, CLIP_EPS, 1 - CLIP_EPS)'
    assert 0.5 <= a.scale <= 2.0, f'c={a.scale} 가 상식 범위 밖'
    assert abs(a.shift) <= 0.05, f'Δ={a.shift} 가 상식 범위 밖'
    if a.scale != 1.0 or a.shift != 0.0:
        lines = (ROOT/'src'/'predict.py').read_text(encoding='utf-8').split(NL)
        idx = [i for i, l in enumerate(lines) if l.strip() == KEY]
        assert len(idx) == 1, f'predict.py 에서 삽입 지점을 {len(idx)}곳 찾았다 (1곳이어야)'
        i = idx[0]
        pad = ' ' * (len(lines[i]) - len(lines[i].lstrip()))
        lines[i:i] = [pad + f'PROB_CENTER = {a.center!r}',
                      pad + f'PROB_SCALE = {a.scale!r}',
                      pad + f'PROB_SHIFT = {a.shift!r}',
                      pad + 'p = PROB_CENTER + PROB_SCALE * (p - PROB_CENTER) + PROB_SHIFT']
        patched = NL.join(lines)
        ast.parse(patched)
        (ROOT/'submit'/'predict.py').write_text(patched, encoding='utf-8')
        build_submit.COPY_FILES = [f for f in build_submit.COPY_FILES if f != 'predict.py']
        hits = build_submit.leakage_scan([ROOT/'submit'/'predict.py'])
        assert not hits, f'패치본에 금지 호출: {hits}'
        print(f'확률공간 아핀 패치: c={a.scale} Δ={a.shift} P0={a.center}  (리키지 검사 통과)')

    build_submit.record_history = lambda *x, **k: print('  이력 기록 건너뜀')
    build_submit.log_submission = lambda *x, **k: print('  SUBMISSIONS.md 기록 건너뜀')
    build_submit.next_zip_path  = lambda m, suffix='': zip_path
    build_submit.main()

    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        pred = z.read('predict.py').decode('utf-8')
    # F전문가는 lgbm_fspec_{seed}.txt 로 저장된다 — 접두어로 갈라야 한다
    n_f = sum(n.startswith('model/lgbm_fspec_') for n in names)
    n_l = sum(n.startswith('model/lgbm_') for n in names) - n_f
    n_m = sum(n.startswith('model/mlp_')  for n in names)
    assert n_l == 20 and n_m == 8, f'LGBM {n_l} · MLP {n_m} (기대 20 · 8)'
    assert n_f >= 1, 'zip 에 fspec 멤버가 없다'
    if a.scale != 1.0 or a.shift != 0.0:
        assert f'PROB_SCALE = {a.scale!r}' in pred, 'zip 의 predict.py 에 c 가 없다'
        assert f'PROB_SHIFT = {a.shift!r}' in pred, 'zip 의 predict.py 에 Δ 가 없다'
        assert '+ PROB_SHIFT' in pred, 'zip 의 predict.py 에 적용 줄이 없다'
    else:
        assert 'PROB_SCALE' not in pred, 'c=1.0·Δ=0 인데 패치 코드가 들어갔다'
        assert 'PROB_SHIFT' not in pred, 'c=1.0·Δ=0 인데 패치 코드가 들어갔다'
    print(f'OK {zip_path.name} alpha={meta["mlp_alpha"]} lgbm={n_l} mlp={n_m} '
          f'fspec={n_f} c={a.scale} d={a.shift:+g} '
          f'{zip_path.stat().st_size/1e6:.1f}MB')
'''), encoding='utf-8')


for tag, spec in SWEEP.items():
    print(f'\n{"="*22} {tag} {"="*22}', flush=True)
    stream([sys.executable, '-u', str(BUILDER),
            '--models', MODELS[tag], '--name', f'p_{tag}',
            '--scale', str(spec.get('_c', 1.0)),
            '--shift', str(spec.get('_d', 0.0)),
            '--center', str(PROB_CENTER)])

## 6. 검증 — 압축을 푼 상태에서 실행

`build_submit.py`는 `script.py`를 `submit/` 작업 디렉터리에서 돌리므로
**`submit/`에는 있고 zip에는 없는 파일**을 못 잡는다. 여기서는 빈 임시 디렉터리에 푼다.

행 독립성 임계값 `TOL = 1e-6` — 어제 측정한 잔차는 MLP BLAS 노이즈이며 α에 정비례했다
(2.1e-08/α). 진짜 누출은 1e-3 이상이라 두 자릿수 이상 떨어져 있다.

In [ ]:
import tempfile, pandas as pd, numpy as np
sys.path.insert(0, str(ROOT/'src')); os.chdir(ROOT)
import build_submit as bs

N_ROWS, TOL = 20000, 1e-6
dummy = bs.make_dummy_test(N_ROWS)
HIDDEN = ('__MACOSX', '.DS_Store', 'Thumbs.db', '.ipynb_checkpoints')

def run_script(work, df):
    d = work/'data'; d.mkdir(exist_ok=True)
    df.to_csv(d/'test.csv', index=False)
    pd.DataFrame({'row_id': df['row_id'], 'control_success': .5}).to_csv(
        d/'sample_submission.csv', index=False)
    shutil.rmtree(work/'output', ignore_errors=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, 'script.py'], cwd=work,
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-1500:]); print(r.stderr[-3000:]); raise SystemExit('script.py 실패')
    return pd.read_csv(work/'output'/'submission.csv'), time.time()-t0

stats = {}
for tag in SWEEP:
    zp = ROOT/f'p_{tag}.zip'
    print(f'\n{"="*22} {zp.name} {"="*22}')
    assert len(zp.name) <= 30
    with zipfile.ZipFile(zp) as z:
        assert z.testzip() is None, 'CRC 불일치'
        names = z.namelist()
    assert not [n for n in names if any(h in n for h in HIDDEN)], '숨은 파일'
    for req in ['script.py', 'requirements.txt']:
        assert req in names, f'루트에 {req} 없음'
    assert 'script.py' in {n.split('/', 1)[0] for n in names}, '최상위 래퍼 폴더가 있다'
    print(f'  구조 정상 · 구성원 {len(names)}개 · 파일명 {len(zp.name)}자')

    with tempfile.TemporaryDirectory() as tmp:
        work = Path(tmp)/'x'; work.mkdir()
        with zipfile.ZipFile(zp) as z: z.extractall(work)
        assert 'torch' not in (work/'requirements.txt').read_text(), 'requirements 에 torch'

        base, el = run_script(work, dummy)
        proj = el/len(dummy)*245789
        assert list(base.columns) == ['row_id','control_success']
        assert len(base) == len(dummy)
        assert base['row_id'].equals(dummy['row_id'].reset_index(drop=True)), 'row_id 불일치'
        p = base['control_success']
        assert p.notna().all() and p.between(0,1).all()
        assert proj < 480, f'추론 시간 여유 부족 ({proj:.0f}s)'
        stats[tag] = {'mean': float(p.mean()), 'std': float(p.std(ddof=0)),
                      'min': float(p.min()), 'max': float(p.max())}
        print(f'  실행 {el:.1f}s -> 245,789행 환산 {proj:.0f}s / 600s ({proj/6:.0f}%)')
        print(f'  출력 정상  mean={p.mean():.6f}  std={p.std(ddof=0):.6f}  '
              f'[{p.min():.4f}, {p.max():.4f}]')

        ref = base.set_index('row_id')['control_success']
        sh, _ = run_script(work, dummy.sample(frac=1, random_state=42).reset_index(drop=True))
        d1 = (sh.set_index('row_id')['control_success'] - ref).abs().max()
        half = len(dummy)//2
        ch = pd.concat([run_script(work, dummy.iloc[:half].reset_index(drop=True))[0],
                        run_script(work, dummy.iloc[half:].reset_index(drop=True))[0]])
        d2 = (ch.set_index('row_id')['control_success'] - ref).abs().max()
        assert d1 < TOL, f'셔플 후 예측이 달라진다 ({d1:.3e}) — 규정 위반 의심'
        assert d2 < TOL, f'청크 분할 예측이 달라진다 ({d2:.3e}) — 규정 위반 의심'
        print(f'  행 독립성  셔플 {d1:.1e} · 청크 {d2:.1e}'
              + ('' if max(d1,d2) == 0 else '  (BLAS 노이즈)'))

print('\n' + '='*54)
print(f'{len(SWEEP)}개 전부 통과. 제출 가능.')
for t, s in stats.items():
    print(f'   p_{t}.zip   mean={s["mean"]:.6f}  std={s["std"]:.6f}')
print('='*54)

# ── c* 역산기 ──────────────────────────────────────────────────────
# BS = Var(p) − 2Cov(p,y) + (p̄−r)² + r(1−r)  (y∈{0,1} 항등식)
#   => c* = Cov/Var = [BSS·r(1−r)/1e5 + Var(p) + (p̄−r)²] / (2·Var(p))
#   이득 = 1e5 · Var(p) · (c*−1)² / r(1−r)
# ⚠️ Var(p) 는 2024 더미 기준 대리값이다. c* 는 "슬롯을 쓸지" 판단용 추정치이고,
#    확정은 LB 2점(현 챔피언 포함 3점)이 한다 — 곡선이 정확히 2차이므로.
def cstar(bss, std_p, pbar, r=0.480):
    V, u = std_p**2, r*(1-r)
    c = (bss*u/1e5 + V + (pbar-r)**2) / (2*V)
    return c, 1e5*V*(c-1)**2/u

print('\nc* 역산 — 제출 후 아래에 LB 점수를 넣어 실행할 것')
print('예:  for r in (0.475, 0.480, 0.486):')
print('         c, g = cstar(1084.14, stats["base"]["std"], stats["base"]["mean"], r)')
print('         print(f"r={r}  c*={c:.3f}  예상이득={g:+.1f}")')

## 7. Drive 저장

결과는 `hyejin:reports/11_constant_sweep_report`에 이어서 기록한다.

In [ ]:
OUT = DRIVE / 'outputs' / 'alpha_probe'
OUT.mkdir(parents=True, exist_ok=True)
for tag in SWEEP:
    src = ROOT / f'p_{tag}.zip'
    shutil.copy2(src, OUT / src.name)
    print(f'{OUT}/{src.name}   ({src.stat().st_size/1e6:.1f} MB)')

print('\n제출할 것:')
for i, t in enumerate(SWEEP, 1):
    print(f'  {i}. p_{t}.zip')
print('\n비교 기준: p_x00.zip = 1088.372727694   (noid 0 · c=1.05, 08-23 챔피언)')
print('  Δ 라운드 판정:  b = -(ΔScore/K_BSS + Δ**2) / (2*Δ)   ->  내일 Δ* = -b')
print('결과는 hyejin:reports/11_constant_sweep_report.md §4-9 에 기록한다.')

## 8. 곡률 스크리너 — 슬롯을 쓰기 전에 축을 고른다

원우 §7.131 의 방법이다. **곡률은 라벨 없이 계산된다.**

```
p(x) ≈ p₀ + (x−x₀)·d ,  d = dp/dx (중심화)
C ≡ K·Var(d)            이득(x*) = C·(x*−x₀)²
```

### ⚠️ 더미 두 개가 필요하다 (2026-08-25 발견)

`make_dummy_test` 는 2024 행에 `season=2025` 라벨을 붙인다. 그런데 `asof`(시즌
진행분) 축의 오프셋은 **2024 종료 시점** 누적이고, 더미 행의 누적값은 2024 *진행 중*
값이라 더 작다.

```
ns = n − offset(2025)      ns > 0 인 행 0.0000%   (중앙값 −528)
```

**시즌 진행분이 전부 음수라 asof 축이 더미에서 통째로 꺼진다.** 증거: LB +26.14 를 낸
배포 축 `gap_success` 의 `C` 가 **0.36** 으로 나왔다. 신호가 없는 게 아니라 못 잰 것이다.

| 축 계열 | 오프셋 | 쓸 더미 |
| --- | --- | --- |
| `hand` · `count4` (ctx) | 안 씀 — 행의 손잡이·카운트만 본다 | `season=2025` (기본) |
| `gap_*` (asof) | **씀** | **`season=2024` 유지** |

`season=2024` 더미는 오프셋이 2023 종료가 되어 `ns > 0` 이다. `season` 피처가 달라
레벨은 어긋나지만, 우리가 재는 것은 **`dp/dβ`** 라 무관하다 (`p(1−p)` 가 0.5% 변할 뿐).

### asof 축은 평균 기여 `m` 도 같이 잰다

새 asof 축을 붙이면 실제 test 평균이 `m` 만큼 움직이는데 **`_d: 'auto'` 는 2025 더미로
재므로 그 이동을 못 본다.** 손실이 `K·m²` 다 (`m=0.003` 이면 −3.6). 그래서 `m` 을 재서
`Δ` 에서 빼 준다.

> 축당 모델 2~3개 + 더미 추론, 약 6분. **슬롯 0개.**

In [ ]:
# ── 곡률 스크리너 (라벨 불필요) ────────────────────────────────────
CHAMP = {'_vw': {'emb4': 0.875, 'noid': 0.125}, '_c': 1.05, '_d': 0.0,
         'cfg': {'FSPEC_ALPHA': 0.458,
                 'CTX_AXES': [['hand', 1.25], ['count4', 1.25]],
                 'ASOF_AXES': [['gap_success', 0.03], ['gap_bsuccess', 0.03]]}}
GBSPEC = {'gap_ball': {'id': 'pitcher_id', 'n_col': 'asof_pitcher_n',
                       'r_col': 'asof_pitcher_ball_rate'}}

def _ctx(h):
    return {'cfg': {'CTX_AXES': [['hand', 1.25], ['count4', 1.25 + h]]}}
def _gb(b):
    s = {'cfg': {'ASOF_AXES': [['gap_success', 0.03], ['gap_bsuccess', 0.03],
                               ['gap_ball', b]]}}
    if b: s['_spec'] = GBSPEC
    return s
def _gs(b):
    return {'cfg': {'ASOF_AXES': [['gap_success', b], ['gap_bsuccess', 0.03]]}}

# 이름 : (섭동, x₀, h, 표시명, 더미)   더미: 'y25'(기본) / 'y24'(asof 축)
SCREEN = {'gs': (_gs, 0.03, 0.015, 'β gap_success [진단·배포축]', 'y24'),
          'gb': (_gb, 0.0,  0.03,  'β gap_ball  (신규 축)',       'y24'),
          'c4': (_ctx, 1.25, 0.25, 'β count4',                    'y25')}

PRED = ROOT / 'screen_pred.py'
PRED.write_text(textwrap.dedent('''
    import os, sys, argparse
    from pathlib import Path
    ap = argparse.ArgumentParser()
    ap.add_argument('--models', required=True); ap.add_argument('--out', required=True)
    ap.add_argument('--dummy', default='y25', choices=['y25', 'y24'])
    a = ap.parse_args()
    ROOT = Path('/content/champion')
    sys.path.insert(0, str(ROOT / 'src')); os.chdir(ROOT)
    import numpy as np, pandas as pd, build_submit as bs, predict as P
    from config import TRAIN_CSV
    if a.dummy == 'y25':
        df = bs.make_dummy_test(245789)                 # season 2025 라벨
    else:
        # asof 축 측정용 — season=2024 를 **유지**해 오프셋이 2023 종료가 되게 한다
        df = pd.read_csv(TRAIN_CSV)
        df = df[df.season == 2024].drop(columns=['control_success']).reset_index(drop=True)
        df['row_id'] = [f'TEST_{i:06d}' for i in range(len(df))]
    p = np.asarray(P.predict_test(df, str(ROOT / a.models)), dtype=float)
    assert np.isfinite(p).all()
    np.save(a.out, p)
    print(f'{a.models} [{a.dummy}] n={len(p):,} mean={p.mean():.6f} std={p.std():.6f}')
'''), encoding='utf-8')

import numpy as np
def merged(pert):
    s = {k: (dict(v) if isinstance(v, dict) else v) for k, v in CHAMP.items()}
    for k, v in pert.items():
        s[k] = {**s.get(k, {}), **v} if k == 'cfg' else v
    s['_d'] = 0.0
    return s

def pred_of(tag, pert, dummy):
    npy = ROOT / f'{tag}_{dummy}.npy'
    if not npy.exists():
        print(f'\n{"=" * 20} {tag} [{dummy}] {"=" * 20}', flush=True)
        md = prepare(tag, merged(pert))
        stream([sys.executable, '-u', str(PRED), '--models', md,
                '--out', str(npy), '--dummy', dummy])
    return np.load(npy)

CS = CHAMP.get('_c', 1.0)
def curv(pa, pb, h):
    d = CS * (pb - pa) / h
    return K_BSS * float(((d - d.mean()) ** 2).mean())

rows = []
for nm, (fn, x0, h, label, dm) in SCREEN.items():
    p0 = pred_of(f'scr_{nm}0', fn(x0), dm)
    lo = pred_of(f'scr_{nm}lo', fn(x0 - h), dm)
    hi = pred_of(f'scr_{nm}hi', fn(x0 + h), dm)
    C  = curv(lo, hi, 2 * h)
    Cf, Cb = curv(p0, hi, h), curv(lo, p0, h)
    asym = abs(Cf - Cb) / max(Cf, Cb, 1e-12)
    # asof 신규 축이면 평균 기여 m 도 잰다 (x₀=0 = 축 없음 기준)
    m = float(CS * (hi - p0).mean()) if x0 == 0 else float('nan')
    rows.append((label, x0, h, C, asym, m, dm))
    ms = f'   평균기여 m={m:+.6f}  ->  Δ 보정 {-m:+.6f} (손실 {K_BSS*m*m:.2f} 회피)' if m == m else ''
    print(f'  {label}: C={C:9.2f}  (전방 {Cf:.2f}/후방 {Cb:.2f}, 비대칭 {asym:.1%}){ms}')

print('\n' + '=' * 84)
print(f'{"축":<30} {"배포":>7} {"C":>10} {"±h":>8} {"±2h":>8} {"비대칭":>7} {"더미":>5}')
print('-' * 84)
for label, x0, h, C, a, m, dm in sorted(rows, key=lambda r: -r[3] * r[2] ** 2):
    print(f'{label:<30} {x0:>7} {C:>10.2f} {C*h**2:>8.3f} {C*(2*h)**2:>8.3f} {a:>7.1%} {dm:>5}')
print('=' * 84)
print('⚠️ 이득 = C·(x*−x₀)² 다. 축마다 h 단위가 다르니 C 만 비교하면 안 된다.')
print('원우 §7.131 눈금:  δ β 공통배율 83.1 · fspec_alpha 22.0 · LGBM wide 0.6(제외)')
print('검산: gap_success 는 LB +26.14 를 낸 배포 축이다. y24 더미에서 C 가 커야 한다.')
print('      (y25 더미에서는 0.36 이었다 — 축이 꺼져 있었다)')

## 9. 제출 게이트 — 슬롯 쓰기 전에 LB 를 예측한다

**홀드아웃은 게이트로 안 쓴다.** 이 축들에서 5전 5패했다 — `MLP_ALPHA`(+8.48) ·
온도(+3.09) · `noid`(+1.85) · `TARGET_PRIOR`(+18.5) · `fspec_alpha`(+2.04) 를 전부
막았을 것이다. **합쳐서 +34 를 놓친다.**

대신 **곡률 기반 예측**을 쓴다. 실적이 있다.

| 제출 | 예측 | 실측 | 오차 |
| --- | ---: | ---: | ---: |
| `lv2` | 1107.049 | 1106.927 | 0.66% |
| `adopt` | 1109.274 | 1109.271 | **0.12%** |

그리고 **라벨이 안 들어간다** — 더미 예측만으로 계산된다.

### 방법

```
BS(β) = BS₀ + (β−β₀)(A₁−2B₁) + (β−β₀)²·Var(d)      d = dp/dβ (중심화)
C ≡ K·Var(d)                     <- 더미에서 계산
u = Δx/2 + ΔScore/(2·C·Δx)       <- 이미 쏜 한 점에서 역산
예측 ΔScore(β) = −C·(β₀−β)·(2u + (β₀−β))
```

**`C` 를 중심차분으로 잰다.** 라운드 8 의 `C = 11.03` 은 `+20%` 전방차분이라
`−` 방향에 외삽 중이었다 (§8-9 정정). 여기서는 `β₀±h` 양쪽을 재서 그 외삽을 없앤다.
비대칭(`C₊` vs `C₋`)도 같이 찍어 곡선이 2차인지 확인한다.

약 5분, **슬롯 0개.** 예측이 챔피언 위일 때만 `## 5. zip 빌드` 로 넘어간다.

In [ ]:
# ── 제출 게이트 ────────────────────────────────────────────────
# 이미 쏜 LB 점 (β_hand 축, 기준선 = 챔피언)
CHAMP_LB   = 1113.0061636942       # p_bh169_1.5 (β_hand = 1.5)
BETA0      = 1.875                 # 곡률·역산의 기준점 (배포 원값). 챔피언과 별개다
LB_PROBES  = {2.0625: 1106.5761424262,   # 08-24 p_bh206
              1.5:    1113.0061636942}   # 08-25 p_bh169_1.5  <- 현 챔피언
GATE_H     = 0.1875                # 중심차분 스텝
MIN_GAIN   = 0.30                  # 이 이상 예측돼야 제출한다

LB_AT_B0 = 1109.2714702849         # β=BETA0 에서의 LB (p_adopt). ΔScore 기준
CAND = SWEEP[list(SWEEP)[0]]
CAND_BETA = dict(CAND['cfg']['CTX_AXES'])['hand']
print(f'후보 β_hand = {CAND_BETA}   (배포 {BETA0})\n')

# ── 0. 예측 스크립트 (셀 21 이 없어도 돌게 자립형으로 둔다) ─────
import numpy as np
PRED = ROOT / 'screen_pred.py'
if not PRED.exists():
    PRED.write_text(textwrap.dedent('''
        import os, sys, argparse
        from pathlib import Path
        ap = argparse.ArgumentParser()
        ap.add_argument('--models', required=True)
        ap.add_argument('--out', required=True)
        a = ap.parse_args()
        ROOT = Path('/content/champion')
        sys.path.insert(0, str(ROOT / 'src')); os.chdir(ROOT)
        import numpy as np, build_submit as bs, predict as P
        df = bs.make_dummy_test(245789)
        p = np.asarray(P.predict_test(df, str(ROOT / a.models)), dtype=float)
        assert np.isfinite(p).all()
        np.save(a.out, p)
        print(f'{a.models}  n={len(p):,}  mean={p.mean():.6f}  std={p.std():.6f}')
    '''), encoding='utf-8')

# ── 1. 세 지점 예측 벡터 확보 (β₀−h, β₀, β₀+h) ──────────────────
# 셋 다 여기서 만든다 — 이전 세션의 캐시에 의존하지 않는다.
P_OF = {}
for b in (BETA0 - GATE_H, BETA0, BETA0 + GATE_H):
    tag = f'gt{str(round(b, 4)).replace(".", "")}'
    npy = ROOT / f'{tag}.npy'
    if not npy.exists():
        spec = {k: (dict(v) if isinstance(v, dict) else v) for k, v in CAND.items()}
        spec['cfg'] = {**spec['cfg'], 'CTX_AXES': [['hand', b], ['count4', 1.25]]}
        spec['_d'] = 0.0          # 게이트는 레벨을 중심화로 처리한다
        print(f'{"=" * 18} β_hand = {b} {"=" * 18}', flush=True)
        md = prepare(tag, spec)
        stream([sys.executable, '-u', str(PRED), '--models', md, '--out', str(npy)])
    P_OF[b] = np.load(npy)

# ── 2. 곡률 — 중심차분 + 양쪽 전방차분 (비대칭 점검) ────────────
CS = CAND.get('_c', 1.0)
def curv(pa, pb, h):
    d = CS * (pb - pa) / h
    return K_BSS * float(((d - d.mean()) ** 2).mean())

C_ctr = curv(P_OF[BETA0 - GATE_H], P_OF[BETA0 + GATE_H], 2 * GATE_H)
C_fwd = curv(P_OF[BETA0], P_OF[BETA0 + GATE_H], GATE_H)
C_bwd = curv(P_OF[BETA0 - GATE_H], P_OF[BETA0], GATE_H)
print(f'\n곡률  중심차분 C = {C_ctr:6.2f}   (전방 {C_fwd:.2f} / 후방 {C_bwd:.2f})')
asym = abs(C_fwd - C_bwd) / max(C_fwd, C_bwd)
print(f'      비대칭 {asym:.1%}' + ('  <- 2차 가정이 흔들린다. 예측을 낮춰 본다'
                                    if asym > 0.25 else '  <- 2차 가정 양호'))

# ── 3. 이미 쏜 점에서 꼭짓점 역산 ──────────────────────────────
print()
for b, lb in LB_PROBES.items():
    dx, ds = b - BETA0, lb - LB_AT_B0
    u = dx / 2 + ds / (2 * C_ctr * dx)
    print(f'프로브 β={b}  ΔScore={ds:+.4f}  ->  u={u:+.4f}   β* = {BETA0 + u:.4f}'
          f'   최적 이득 {C_ctr * u * u:+.2f}')
U = dx / 2 + ds / (2 * C_ctr * dx)

# ── 4. 후보 예측 ───────────────────────────────────────────────
d_c = BETA0 - CAND_BETA
pred = -C_ctr * d_c * (2 * U + d_c)          # β=BETA0 기준 ΔScore
print(f'\n{"=" * 62}')
print(f'후보 β_hand = {CAND_BETA}   예측 ΔScore = {pred:+.3f}')
print(f'예측 LB = {LB_AT_B0 + pred:.4f}    (챔피언 {CHAMP_LB})')
print(f'챔피언 대비 = {LB_AT_B0 + pred - CHAMP_LB:+.3f}')
print(f'손익분기 C = {(ds / (2 * dx)) / (-dx / 2 - d_c / 2):.1f}   실측 C = {C_ctr:.2f}')
print('=' * 62)
pred_vs_champ = LB_AT_B0 + pred - CHAMP_LB
if pred_vs_champ >= MIN_GAIN:
    print(f'✅ 제출한다 — 챔피언 대비 예측 {pred_vs_champ:+.2f} ≥ {MIN_GAIN}')
elif pred_vs_champ > 0:
    print(f'△ 이득이 작다 ({pred_vs_champ:+.2f}). 다른 β 를 넣어 다시 돌려 볼 것')
else:
    print(f'⛔ 보류 — 예측이 챔피언 아래다 ({pred_vs_champ:+.2f}). 슬롯을 쓰지 않는다')

# ── 4-b. LB 점이 2개 이상이면 **포물선을 직접 적합**한다 ────────
#   라벨없는 C 와 LB 가 말하는 C 를 대조하는 자리다. 둘이 맞으면 모델이 확정된다.
if len(LB_PROBES) >= 2:
    bs_ = np.array([BETA0] + list(LB_PROBES))
    lb_ = np.array([LB_AT_B0] + [LB_PROBES[b] for b in LB_PROBES])
    a2, a1, a0 = np.polyfit(bs_, lb_, 2)
    C_fit, B_fit = -a2, -a1 / (2 * a2)
    print(f'\n{"=" * 62}')
    print(f'LB {len(bs_)}점 포물선 직접 적합  (β: {", ".join(f"{b:g}" for b in bs_)})')
    print(f'  C_LB  = {C_fit:7.2f}   vs  라벨없는 C_중심 = {C_ctr:7.2f}'
          f'   비 {C_fit / C_ctr:.3f}')
    print(f'  β*_LB = {B_fit:7.4f}   vs  한 점 역산 β*   = {BETA0 + U:7.4f}')
    print(f'  β*_LB 에서의 이득 = {C_fit * (B_fit - BETA0) ** 2:+.2f}'
          f'   ->  LB {LB_AT_B0 + C_fit * (B_fit - BETA0) ** 2:.2f}'
          f'   (챔피언 대비 {LB_AT_B0 + C_fit * (B_fit - BETA0) ** 2 - CHAMP_LB:+.2f})')
    if abs(C_fit / C_ctr - 1) < 0.15:
        print('  ✅ 두 C 가 15% 안에서 일치 — 2차 모델 확정. 다음 한 장은 **보간**이다.')
    else:
        print('  ⚠️ 두 C 가 어긋난다. 2차 가정이 그 구간에서 깨졌을 수 있다.')
        print('     라벨없는 C 는 β₀ 근방의 곡률이고, C_LB 는 전 구간 평균이다.')
    if len(bs_) >= 4:
        r = lb_ - np.polyval([a2, a1, a0], bs_)
        print(f'  잔차 max {np.abs(r).max():.4f}  (4점 이상이면 2차 적합도 점검)')
    print(f'{"=" * 62}')
    print('  적합값으로 다시 본 후보들:')
    for b in (1.75, 1.625, 1.5, 1.375, 1.25, round(B_fit, 4)):
        v = C_fit * ((B_fit - BETA0) ** 2 - (b - B_fit) ** 2)
        print(f'    β={b:<8} 예측 {v:+7.3f}   LB {LB_AT_B0 + v:.2f}'
              f'   챔피언 대비 {LB_AT_B0 + v - CHAMP_LB:+7.3f}')

# ── 5. 최적 β 와, 그 값을 쐈을 때의 예측 (한 점 역산 기준) ──────
print(f'\n참고(한 점 역산): 예측 최적 β* = {BETA0 + U:.4f} 를 쏘면 {C_ctr * U * U:+.2f}'
      f'  ->  LB {LB_AT_B0 + C_ctr * U * U:.2f}')
print('  ⚠️ 그건 한 점 역산이라 C 오차가 그대로 실린다. 3점이 모인 뒤에 간다.')
for b in (1.75, 1.6875, 1.625, 1.5, 1.375):
    d = BETA0 - b
    print(f'    β={b:<7} 예측 {-C_ctr * d * (2 * U + d):+7.3f}   LB {LB_AT_B0 - C_ctr * d * (2 * U + d):.2f}')